# Notebook 10: UMAP — Uniform Manifold Approximation and Projection

**Difficulty:** ⭐⭐⭐⭐ | **Estimated Time:** 60 minutes

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Explain UMAP's conceptual foundation (topological data analysis) at a high level — without needing category theory
2. Describe the role of `n_neighbors`, `min_dist`, and `metric` hyperparameters
3. Compare UMAP vs. t-SNE on: speed, global structure preservation, and out-of-sample embedding
4. Apply UMAP both as a visualisation tool and as a feature transformer for downstream models

---

**Week 11 Theme:** Customer Segmentation / Retail Data

**Prerequisites:** PCA (Notebook 07), t-SNE (Notebook 09), KMeans (Notebooks 01-03)

## The Analogy — Before Any Math

**t-SNE** is like a cartographer who's obsessed with making sure neighbouring villages always look close on the map. They'll happily move entire continents around to achieve this — the global layout is sacrificed for local accuracy.

**UMAP** is like t-SNE's faster, more globally-aware cousin. It uses a smarter map-making strategy:

> *"Make sure neighbouring towns are near each other AND keep countries in roughly the right continents."*

UMAP tries to preserve **both local neighbourhood structure AND the overall topology (shape) of the data**. It's as if you made a map that keeps your street accurate, keeps your neighbourhood accurate, and also keeps Europe in the right place relative to Asia.

The result: UMAP produces visualisations that look like t-SNE (clear local clusters) but are **faster** to compute, **more stable**, and preserve **more global structure** — so distances between clusters are somewhat more meaningful.

Importantly, UMAP can also **embed new points** into an existing layout — something t-SNE fundamentally cannot do.

## Why Does This Matter for ML?

UMAP has rapidly become a standard tool in the ML toolkit, increasingly replacing t-SNE in many settings:

| Advantage over t-SNE | Practical impact |
|----------------------|------------------|
| 5-10× faster | Can run on 100k+ points in minutes |
| Preserves global structure | Cluster distances carry some meaning |
| Has `transform()` method | Can embed new customers into existing map |
| Works as feature extractor | UMAP → KMeans pipeline is viable |
| Multiple metrics supported | Cosine distance for text/NLP; Hamming for binary data |

**Real-world applications at Week 11's retail theme:**
- Embed new customers into an existing customer map (online, as they arrive)
- Build a UMAP-reduced feature space for faster KMeans clustering
- Visualise how product categories cluster in purchase-behaviour space
- Detect anomalous customers (those that don't fit any cluster in UMAP space)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_digits, fetch_20newsgroups
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.feature_extraction.text import TfidfVectorizer
import time
import warnings

warnings.filterwarnings('ignore')      # suppress convergence and deprecation warnings
np.random.seed(42)                      # global reproducibility seed
sns.set_theme(style='whitegrid')        # consistent seaborn visual style

# ── Try to import umap-learn (optional dependency)
try:
    import umap                         # umap-learn package — pip install umap-learn
    UMAP_AVAILABLE = True
    print(f"umap-learn is available.")
except ImportError:
    UMAP_AVAILABLE = False
    print("umap-learn not installed. Run: pip install umap-learn")
    print("We will demonstrate concepts using t-SNE as a proxy where UMAP is unavailable.")
    print("All code cells are written to handle both cases gracefully.")

print(f"\nNumPy {np.__version__} | UMAP available: {UMAP_AVAILABLE}")

## UMAP's Conceptual Foundation (High Level)

UMAP's full theoretical framework comes from **topological data analysis** — specifically, something called *fuzzy simplicial sets*. You don't need to understand the mathematics to use UMAP effectively, but here's an intuitive walkthrough of the two-step process:

### Step 1 — Build a weighted graph in high-D

For each point, UMAP finds its `n_neighbors` nearest neighbours and assigns **edge weights** based on distance:
- Close neighbours get weight close to 1 ("very likely to be connected")
- Distant points get weight close to 0 ("probably not connected")

This creates a **fuzzy neighbourhood graph** — a network where every point is connected to its local neighbours with varying degrees of certainty.

The key insight: UMAP normalises these weights per-point so that the *closest* neighbour always has weight ~1.0, regardless of the absolute scale. This makes UMAP robust to varying density across the dataset.

### Step 2 — Optimise a low-D layout that preserves the graph

UMAP then places points in 2D space and optimises their positions to match the high-D graph as closely as possible. It does this by:
- **Attracting** connected points (high edge weight → pull together)
- **Repelling** unconnected points (low edge weight → push apart)

This is similar to t-SNE's KL divergence minimisation, but uses **cross-entropy** instead, which treats attraction and repulsion more symmetrically.

### Key difference from t-SNE

| | t-SNE | UMAP |
|-|-------|------|
| Similarity model | Gaussian probabilities | Fuzzy simplicial sets |
| Optimisation | KL divergence | Cross-entropy |
| Normalisation | Global (all $p_{ij}$ sum to 1) | Local per-point |
| Result | Local clusters, distorted global | Local + better global |

The local normalisation in UMAP is why it preserves global structure better: points in dense regions aren't artificially "inflated" relative to points in sparse regions.

In [ ]:
np.random.seed(42)

# ── Load the digits dataset as our benchmark
digits = load_digits()
X_digits = digits.data       # (1797, 64) — flattened 8x8 grayscale images
y_digits = digits.target     # (1797,)    — digit labels 0-9

# Standardise before any dimensionality reduction
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_digits)

print(f"Digits dataset: {X_scaled.shape[0]} samples, {X_scaled.shape[1]} features")
print()

# ── Compute UMAP or fall back to t-SNE
if UMAP_AVAILABLE:
    print("Running UMAP (n_neighbors=15, min_dist=0.1)...")
    t0 = time.time()
    reducer_umap = umap.UMAP(n_components=2,
                             n_neighbors=15,   # how many local neighbours to consider
                             min_dist=0.1,     # how tightly to pack points in 2D
                             random_state=42)
    X_umap = reducer_umap.fit_transform(X_scaled)
    time_umap = time.time() - t0
    print(f"UMAP completed in {time_umap:.2f}s")
else:
    print("UMAP not available — using t-SNE (perplexity=30) as proxy for right panel.")
    t0 = time.time()
    X_umap = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(X_scaled)
    time_umap = time.time() - t0
    print(f"t-SNE proxy completed in {time_umap:.2f}s")

# ── Also run t-SNE for the left panel comparison
print("Running t-SNE (perplexity=30) for comparison...")
t0 = time.time()
X_tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000).fit_transform(X_scaled)
time_tsne = time.time() - t0
print(f"t-SNE completed in {time_tsne:.2f}s")

# ── Side-by-side comparison
cmap = plt.cm.get_cmap('tab10', 10)
right_title = 'UMAP (n_neighbors=15)' if UMAP_AVAILABLE else 't-SNE proxy (UMAP not installed)'

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, X_2d, title, elapsed in zip(
        axes,
        [X_tsne, X_umap],
        [f't-SNE (perplexity=30)  [{time_tsne:.1f}s]', f'{right_title}  [{time_umap:.1f}s]']):
    sc = ax.scatter(X_2d[:, 0], X_2d[:, 1],
                    c=y_digits, cmap=cmap, s=10, alpha=0.8)
    ax.set_title(title, fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])

fig.colorbar(sc, ax=axes, label='Digit class', fraction=0.025, pad=0.04)
fig.suptitle('t-SNE vs. UMAP on Digits Dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print()
if UMAP_AVAILABLE:
    speedup = time_tsne / time_umap if time_umap > 0 else float('inf')
    print(f"Speed comparison: t-SNE = {time_tsne:.1f}s, UMAP = {time_umap:.1f}s")
    print(f"UMAP is approximately {speedup:.1f}x faster on this dataset.")

## Key Hyperparameters

UMAP has three main hyperparameters. Understanding them lets you tune UMAP for your specific use case.

### 1. `n_neighbors` (default: 15)

Controls **how many nearest neighbours** each point considers when building the high-D graph. This is UMAP's analogue of t-SNE's perplexity.

| `n_neighbors` | Effect |
|---------------|--------|
| 5 | Very local focus — fine-grained micro-structure, may fragment real clusters |
| 15 (default) | Good balance of local and global structure |
| 50 | More global — smoother layout, better inter-cluster relationships |
| 200 | Very global — almost a PCA-like linear projection |

### 2. `min_dist` (default: 0.1)

Controls **how tightly points are packed** in the 2D output. It's the minimum distance allowed between points in the low-dimensional representation.

| `min_dist` | Effect |
|------------|--------|
| 0.001 | Very tight clusters — good for identifying discrete clusters |
| 0.1 (default) | Balanced — clusters are distinct but not over-squished |
| 0.5 | More spread out — better for seeing continuous gradients |
| 0.99 | Points spread uniformly — loses cluster structure |

### 3. `metric` (default: 'euclidean')

Which distance function to use when building the neighbourhood graph. Unlike t-SNE (which only supports Euclidean), UMAP supports:
- `'euclidean'` — standard L2 distance (numeric features)
- `'cosine'` — angle-based similarity (great for text embeddings, unit-norm vectors)
- `'manhattan'` — L1 distance (sparse, count data)
- `'hamming'` — fraction of differing bits (binary data)
- Many others: `'correlation'`, `'jaccard'`, `'canberra'`, custom functions

In [ ]:
np.random.seed(42)

# ── n_neighbors comparison: how local vs. global structure changes
n_neighbors_values = [5, 15, 50, 200]
nn_results = {}

if UMAP_AVAILABLE:
    print("Running UMAP with different n_neighbors values...")
    for nn in n_neighbors_values:
        print(f"  n_neighbors={nn}...")
        reducer = umap.UMAP(n_components=2, n_neighbors=nn,
                            min_dist=0.1, random_state=42)
        nn_results[nn] = reducer.fit_transform(X_scaled)
else:
    # t-SNE proxy: map n_neighbors → roughly equivalent perplexity
    # (perplexity ≈ n_neighbors / 3 is a rough rule of thumb)
    proxy_perplexities = {5: 5, 15: 15, 50: 30, 200: 50}
    print("Running t-SNE proxy with varying perplexity (proxy for n_neighbors)...")
    for nn, perp in proxy_perplexities.items():
        print(f"  n_neighbors={nn} → t-SNE perplexity={perp}...")
        tsne_proxy = TSNE(n_components=2, perplexity=perp, random_state=42, n_iter=500)
        nn_results[nn] = tsne_proxy.fit_transform(X_scaled)

# ── Plot 4 subplots
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
cmap = plt.cm.get_cmap('tab10', 10)

for ax, nn in zip(axes, n_neighbors_values):
    X_2d = nn_results[nn]
    ax.scatter(X_2d[:, 0], X_2d[:, 1],
               c=y_digits, cmap=cmap, s=8, alpha=0.8)
    label = f'n_neighbors={nn}' if UMAP_AVAILABLE else f'n_neighbors={nn}\n(t-SNE proxy)'
    ax.set_title(label, fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])

algo_name = 'UMAP' if UMAP_AVAILABLE else 't-SNE (proxy for UMAP)'
fig.suptitle(f'Effect of n_neighbors on {algo_name} Layout (Digits)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print()
print("What you should observe:")
print("  n_neighbors=5:   Fragmented micro-structure — many small islands within each digit")
print("  n_neighbors=15:  Clean clusters — similar digit classes may appear adjacent (good global structure)")
print("  n_neighbors=50:  Smoother layout — fewer isolated fragments, more connectivity")
print("  n_neighbors=200: Global structure dominates — similar to PCA, clusters may merge")

In [ ]:
np.random.seed(42)

# ── min_dist comparison: cluster tightness vs. spread
min_dist_values = [0.001, 0.1, 0.5, 0.99]
md_results = {}

if UMAP_AVAILABLE:
    print("Running UMAP with different min_dist values (n_neighbors=15 fixed)...")
    for md in min_dist_values:
        print(f"  min_dist={md}...")
        reducer = umap.UMAP(n_components=2, n_neighbors=15,
                            min_dist=md, random_state=42)
        md_results[md] = reducer.fit_transform(X_scaled)
else:
    # Without UMAP we use t-SNE with same layout and note that min_dist isn't directly available
    print("UMAP not available — showing single t-SNE for reference (min_dist is a UMAP-only parameter).")
    base = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=500).fit_transform(X_scaled)
    for md in min_dist_values:
        md_results[md] = base   # same result for all — just for code compatibility

# ── Plot 4 subplots
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
cmap = plt.cm.get_cmap('tab10', 10)

for ax, md in zip(axes, min_dist_values):
    X_2d = md_results[md]
    ax.scatter(X_2d[:, 0], X_2d[:, 1],
               c=y_digits, cmap=cmap, s=8, alpha=0.8)
    label = f'min_dist={md}' if UMAP_AVAILABLE else f'min_dist={md}\n(requires UMAP)'
    ax.set_title(label, fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])

algo_name = 'UMAP' if UMAP_AVAILABLE else 't-SNE (install umap-learn to see min_dist effect)'
fig.suptitle(f'Effect of min_dist on {algo_name} Layout (Digits)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print()
if UMAP_AVAILABLE:
    print("What you should observe:")
    print("  min_dist=0.001:  Very tight, compact clusters — best for identifying discrete groups")
    print("  min_dist=0.1:    Balanced — clusters are distinct but show some internal spread")
    print("  min_dist=0.5:    Points spread within clusters — reveals continuous gradients")
    print("  min_dist=0.99:   Uniform spread — clusters nearly dissolve, structure is lost")
else:
    print("Note: min_dist is a UMAP-specific parameter with no t-SNE equivalent.")
    print("Install umap-learn (pip install umap-learn) to see its effect on layout tightness.")

## UMAP Speed vs. t-SNE

One of UMAP's most compelling advantages is speed. Here's a theoretical comparison:

| Algorithm | Time complexity | Memory | 10k points | 100k points |
|-----------|----------------|--------|-----------|-------------|
| Naive t-SNE | O(n²) | O(n²) | ~30 min | impractical |
| Barnes-Hut t-SNE | O(n² log n) | O(n) | ~3 min | ~60 min |
| **UMAP** | **O(n^1.14) empirical** | O(n) | **~15 sec** | **~2 min** |

Why is UMAP faster? Several reasons:

1. **Approximate nearest neighbours** — UMAP uses optimised ANN algorithms (NN-Descent) rather than exact O(n²) distance computations
2. **Sparse graph** — UMAP only stores `n_neighbors` edges per point, not all n² pairs
3. **Stochastic gradient descent** — UMAP uses SGD for optimisation, which converges faster than t-SNE's gradient descent on the full dataset
4. **Fewer iterations** — UMAP typically needs 200–500 epochs vs. t-SNE's 1000+

In [ ]:
np.random.seed(42)

# ── Benchmark t-SNE vs. UMAP at different dataset sizes
# Note: we use small sizes to keep the demo fast; scaling trends are clear even at n=5000
sample_sizes = [500, 1000, 1797]   # limited to full digits dataset (1797 points)

times_tsne = []
times_umap = []

for n in sample_sizes:
    # Take a stratified subsample to keep class balance
    idx = np.random.choice(len(X_scaled), size=min(n, len(X_scaled)), replace=False)
    X_bench = X_scaled[idx]

    # Time t-SNE (n_iter=500 for speed in demo)
    t0 = time.time()
    TSNE(n_components=2, perplexity=30, random_state=42, n_iter=500).fit_transform(X_bench)
    times_tsne.append(time.time() - t0)

    # Time UMAP (or proxy)
    t0 = time.time()
    if UMAP_AVAILABLE:
        umap.UMAP(n_components=2, random_state=42).fit_transform(X_bench)
    else:
        # t-SNE with fewer iterations as a rough proxy
        TSNE(n_components=2, perplexity=30, random_state=0, n_iter=300).fit_transform(X_bench)
    times_umap.append(time.time() - t0)

    print(f"n={n:5d}: t-SNE={times_tsne[-1]:.2f}s  {'UMAP' if UMAP_AVAILABLE else 'tSNE-proxy'}={times_umap[-1]:.2f}s")

# ── Plot scaling curves
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(sample_sizes, times_tsne, 'o-', color='steelblue',
        linewidth=2, markersize=8, label='t-SNE (n_iter=500)')
umap_label = 'UMAP' if UMAP_AVAILABLE else 't-SNE proxy (install umap-learn)'
ax.plot(sample_sizes, times_umap, 's-', color='coral',
        linewidth=2, markersize=8, label=umap_label)

ax.set_xlabel('Number of data points (n)', fontsize=12)
ax.set_ylabel('Wall-clock time (seconds)', fontsize=12)
ax.set_title('Scaling Comparison: t-SNE vs. UMAP', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)

# Annotate each point with its time
for n, t_t, t_u in zip(sample_sizes, times_tsne, times_umap):
    ax.annotate(f'{t_t:.1f}s', (n, t_t), textcoords='offset points',
                xytext=(5, 5), fontsize=9, color='steelblue')
    ax.annotate(f'{t_u:.1f}s', (n, t_u), textcoords='offset points',
                xytext=(5, -15), fontsize=9, color='coral')

plt.tight_layout()
plt.show()

if UMAP_AVAILABLE and len(sample_sizes) > 1:
    ratios = [t / u for t, u in zip(times_tsne, times_umap)]
    print(f"\nSpeed ratios (t-SNE time / UMAP time): {[f'{r:.1f}x' for r in ratios]}")
    print("UMAP advantage grows with dataset size.")

## Global Structure Preservation: A Key UMAP Advantage

This is where UMAP genuinely outperforms t-SNE. Recall that in t-SNE, **inter-cluster distances are meaningless** — you cannot say that Cluster A is more similar to Cluster B than to Cluster C based on where they appear in the plot.

With UMAP, the story is different:

> **UMAP's cluster distances are *somewhat* meaningful.** Clusters that are farther apart in UMAP space tend to be more different in the original space.

This is because UMAP's local normalisation (per-point edge weights) and its cross-entropy objective maintain a better connection between high-D topology and low-D layout.

**Important caveat:** UMAP still distorts global structure — just *less* than t-SNE. Don't treat UMAP distances as exact. Use them for qualitative guidance only.

**Practical implication for retail segmentation:**
- In UMAP space, "Budget Shoppers" and "Churned Customers" might appear far from "Premium Buyers"
- This spatial separation gives you a rough sense of which segments are most different — unlike t-SNE where this would be meaningless

In [ ]:
np.random.seed(42)

# ── Demonstrate global structure preservation on digits dataset
# Hypothesis: visually similar digits (3 vs 8, 4 vs 9) should be closer in UMAP space
# than visually different digits (0 vs 8, 1 vs 8)

if UMAP_AVAILABLE:
    reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42)
    X_embed = reducer.fit_transform(X_scaled)
    embed_name = 'UMAP'
else:
    X_embed = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000).fit_transform(X_scaled)
    embed_name = 't-SNE (proxy — UMAP not installed)'

# Compute centroid of each digit class in the 2D embedding
centroids = {}
for digit in range(10):
    mask = y_digits == digit
    centroids[digit] = X_embed[mask].mean(axis=0)   # mean position of all points of this digit

# Compute inter-centroid distances for selected pairs
digit_pairs = [(3, 8), (4, 9), (0, 8), (1, 8), (0, 1), (5, 6)]
pair_labels = ['3 vs 8\n(similar)', '4 vs 9\n(similar)',
               '0 vs 8\n(different)', '1 vs 8\n(different)',
               '0 vs 1\n(different)', '5 vs 6\n(medium)']

distances = []
for (a, b) in digit_pairs:
    dist = np.linalg.norm(centroids[a] - centroids[b])   # Euclidean distance between centroids
    distances.append(dist)

# Plot: bar chart of inter-centroid distances + the embedding
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: UMAP/t-SNE embedding coloured by digit
cmap = plt.cm.get_cmap('tab10', 10)
sc = axes[0].scatter(X_embed[:, 0], X_embed[:, 1],
                     c=y_digits, cmap=cmap, s=8, alpha=0.75)
# Mark centroids
for digit, c in centroids.items():
    axes[0].text(c[0], c[1], str(digit), fontsize=13, fontweight='bold',
                 ha='center', va='center',
                 bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))
axes[0].set_title(f'{embed_name} Embedding\n(numbers = digit centroids)', fontsize=11)
axes[0].set_xticks([])
axes[0].set_yticks([])

# Right: bar chart of centroid distances
bar_colors = ['steelblue' if 'similar' in lbl else 'coral' if 'different' in lbl else 'goldenrod'
              for lbl in pair_labels]
axes[1].bar(pair_labels, distances, color=bar_colors, edgecolor='white', linewidth=0.5)
axes[1].set_ylabel('Euclidean distance between centroids', fontsize=11)
axes[1].set_title(f'Inter-class Distance in {embed_name} Space\n'
                  '(blue=visually similar, red=visually different)',
                  fontsize=11)
axes[1].tick_params(axis='x', labelsize=9)

plt.tight_layout()
plt.show()

print()
if UMAP_AVAILABLE:
    sim_pairs_avg  = np.mean([distances[0], distances[1]])   # 3vs8, 4vs9
    diff_pairs_avg = np.mean([distances[2], distances[3]])   # 0vs8, 1vs8
    print(f"Avg distance for visually similar pairs (3v8, 4v9):   {sim_pairs_avg:.2f}")
    print(f"Avg distance for visually different pairs (0v8, 1v8): {diff_pairs_avg:.2f}")
    if diff_pairs_avg > sim_pairs_avg:
        print("Result: Visually different digits ARE further apart in UMAP space — global structure preserved!")
    else:
        print("Note: Results may vary — try n_neighbors=30 for stronger global structure.")

## UMAP as a Feature Transformer (Unlike t-SNE)

This is perhaps UMAP's most practically important advantage over t-SNE:

> **UMAP has a `transform()` method.** After fitting on training data, you can project *new* data points into the same 2D (or n-D) space.

t-SNE has no equivalent — it's a fundamentally transductive algorithm. You cannot call `tsne.transform(X_new)`.

### What this enables:

1. **Production embedding:** Fit UMAP on historical customer data, then continuously embed new customers into the same map for monitoring

2. **Feature preprocessing:** Use UMAP to reduce dimensions before classification or clustering. Because you can `transform()` new data, the pipeline generalises to unseen examples

3. **Anomaly detection:** New points that land far from any existing cluster in UMAP space may be anomalies

### Caveats:

- The `transform()` in UMAP is approximate — the new point is embedded by finding its nearest neighbours in the training set and interpolating. It's not as accurate as refitting.
- For very different new data distributions, the embedding may be unreliable.
- UMAP as a feature extractor for classification is more reliable than t-SNE, but PCA or kernel PCA is often simpler and more stable.

In [ ]:
np.random.seed(42)

# ── UMAP → KMeans pipeline
# Compare cluster quality: KMeans on original 64-D vs. KMeans on UMAP 2-D
from sklearn.model_selection import train_test_split

# Split into train/test to properly evaluate the transform() method
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_digits, test_size=0.3, random_state=42, stratify=y_digits
)

n_clusters = 10   # one cluster per digit class

# ── Method 1: KMeans directly on 64-D original features
km_orig = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
km_orig.fit(X_train)
labels_orig_test = km_orig.predict(X_test)
sil_orig = silhouette_score(X_test, labels_orig_test, sample_size=500, random_state=42)

# ── Method 2: UMAP(2) → KMeans on 2-D
if UMAP_AVAILABLE:
    print("Fitting UMAP on training data...")
    umap_reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42)
    X_train_umap = umap_reducer.fit_transform(X_train)     # fit on training set
    X_test_umap  = umap_reducer.transform(X_test)          # transform new test points

    km_umap = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    km_umap.fit(X_train_umap)
    labels_umap_test = km_umap.predict(X_test_umap)
    sil_umap = silhouette_score(X_test_umap, labels_umap_test, random_state=42)

    print(f"\nKMeans on original 64-D:  silhouette = {sil_orig:.4f}")
    print(f"KMeans on UMAP 2-D:       silhouette = {sil_umap:.4f}")
    print()
    print("Note: silhouette scores are computed in DIFFERENT spaces, so they are not directly")
    print("comparable numerically — but both are positive and indicate cluster separation.")
    print("The UMAP pipeline also enables embedding new customers at inference time.")

    # ── Visualise UMAP 2-D space with KMeans cluster assignments
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    cmap10 = plt.cm.get_cmap('tab10', 10)

    # Left: colour by true digit label
    sc = axes[0].scatter(X_test_umap[:, 0], X_test_umap[:, 1],
                         c=y_test, cmap=cmap10, s=15, alpha=0.8)
    axes[0].set_title('UMAP 2-D: True Digit Labels (test set)', fontsize=11)
    axes[0].set_xticks([])
    axes[0].set_yticks([])

    # Right: colour by KMeans cluster assignment
    axes[1].scatter(X_test_umap[:, 0], X_test_umap[:, 1],
                    c=labels_umap_test, cmap=cmap10, s=15, alpha=0.8)
    axes[1].set_title(f'UMAP 2-D: KMeans Clusters (silhouette={sil_umap:.3f})', fontsize=11)
    axes[1].set_xticks([])
    axes[1].set_yticks([])

    fig.colorbar(sc, ax=axes, label='Digit / Cluster', fraction=0.025, pad=0.04)
    fig.suptitle('UMAP as Feature Transformer: fit on train, transform on test',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

else:
    print("UMAP not available — skipping UMAP→KMeans pipeline demo.")
    print(f"KMeans on original 64-D: silhouette = {sil_orig:.4f}")
    print("Install umap-learn to see the full comparison.")

In [ ]:
np.random.seed(42)

# ── Apply to retail synthetic data (Week 11 theme)
# Recreate the same synthetic customer dataset from Notebook 09

segment_params = {
    'Budget Shoppers':   {'recency': 15,  'frequency': 8,  'monetary': 120,  'age': 45, 'online_ratio': 0.2},
    'Loyal Regulars':    {'recency': 5,   'frequency': 20, 'monetary': 350,  'age': 38, 'online_ratio': 0.5},
    'Premium Buyers':    {'recency': 8,   'frequency': 12, 'monetary': 900,  'age': 52, 'online_ratio': 0.4},
    'Digital Natives':   {'recency': 3,   'frequency': 25, 'monetary': 280,  'age': 28, 'online_ratio': 0.95},
    'Churned Customers': {'recency': 90,  'frequency': 2,  'monetary': 80,   'age': 55, 'online_ratio': 0.15},
}
n_per_seg = 120

all_X, all_y = [], []
for seg, p in segment_params.items():
    seg_data = np.column_stack([
        np.random.normal(p['recency'],      p['recency'] * 0.3,      n_per_seg),
        np.random.normal(p['frequency'],    p['frequency'] * 0.25,   n_per_seg),
        np.random.normal(p['monetary'],     p['monetary'] * 0.3,     n_per_seg),
        np.random.normal(p['age'],          5,                        n_per_seg),
        np.clip(np.random.normal(p['online_ratio'], 0.1, n_per_seg), 0, 1),
    ])
    all_X.append(seg_data)
    all_y.extend([seg] * n_per_seg)

X_retail  = np.vstack(all_X)
y_retail  = np.array(all_y)
X_retail_scaled = StandardScaler().fit_transform(X_retail)

# ── Compute all three projections
print("Computing PCA projection...")
X_retail_pca  = PCA(n_components=2, random_state=42).fit_transform(X_retail_scaled)

print("Computing t-SNE projection...")
X_retail_tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000).fit_transform(X_retail_scaled)

if UMAP_AVAILABLE:
    print("Computing UMAP projection...")
    umap_retail   = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42)
    X_retail_umap = umap_retail.fit_transform(X_retail_scaled)
    umap_label = 'UMAP'
else:
    print("UMAP not available — using second t-SNE run as proxy for UMAP panel.")
    X_retail_umap = TSNE(n_components=2, perplexity=30, random_state=1, n_iter=1000).fit_transform(X_retail_scaled)
    umap_label = 't-SNE proxy (install umap-learn)'

# ── Three-panel comparison
seg_names  = list(segment_params.keys())
seg_colors = dict(zip(seg_names, plt.cm.Set1.colors[:5]))

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, X_2d, title in zip(
        axes,
        [X_retail_pca, X_retail_tsne, X_retail_umap],
        ['PCA (2D)', 't-SNE (perplexity=30)', umap_label]):
    for seg in seg_names:
        mask = y_retail == seg
        ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
                   c=[seg_colors[seg]], label=seg, s=15, alpha=0.75)
    ax.set_title(title, fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])

axes[2].legend(loc='lower left', fontsize=8, markerscale=2, framealpha=0.9)
fig.suptitle('Customer Segmentation: PCA vs. t-SNE vs. UMAP', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print()
print("Compare the three projections:")
print("  PCA:   Linear — may miss non-linear cluster boundaries")
print("  t-SNE: Excellent local clusters but inter-cluster distances meaningless")
print(f"  {umap_label}: Local clusters + better global structure preservation")

## When to Use Which Tool

Here's a practical decision guide for choosing between PCA, t-SNE, and UMAP:

```
                           ┌─────────────────────────────────────────────┐
                           │          Dimensionality Reduction            │
                           └──────────────────┬──────────────────────────┘
                                              │
                           ┌──────────────────▼──────────────────────────┐
                           │  Do you need to embed new points later?     │
                           └───────────┬─────────────────────┬───────────┘
                                 Yes   │                 No  │
                                       │                     │
                    ┌──────────────────▼──┐       ┌──────────▼──────────────┐
                    │  Is n > 100,000?   │       │  Visualization only?   │
                    └───┬────────────┬───┘       └──────────┬──────────────┘
                    Yes │        No  │                       │
                        │            │                 Yes   │   No
                    ┌───▼──┐   ┌────▼──────┐               │
                    │ PCA  │   │   UMAP    │     ┌──────────▼──────────┐
                    │(fast,│   │(transform │     │  n < 100k? → t-SNE │
                    │exact)│   │ capable)  │     │  n >= 100k? → UMAP  │
                    └──────┘   └───────────┘     └─────────────────────┘
```

### Summary table

| Use case | Best choice | Why |
|----------|-------------|-----|
| Reduce dims for linear model | PCA | Interpretable, fast, stable |
| Reduce dims for tree-based model | PCA or none | Trees handle high-D natively |
| Visualise n < 100k, local clusters | t-SNE | Gold standard for local structure |
| Visualise n > 100k | UMAP | t-SNE too slow |
| Preserve global structure | UMAP > t-SNE >> meaningless | |
| Embed new points at inference time | UMAP | t-SNE cannot do this |
| NLP embeddings (cosine distance) | UMAP with `metric='cosine'` | Cosine is natural for unit vectors |
| Streaming / online learning | PCA (incremental PCA) | UMAP/t-SNE are batch algorithms |

In [ ]:
np.random.seed(42)

# ── UMAP with cosine metric on a tiny text corpus
# Topic-labelled sentences — cosine distance is natural for TF-IDF vectors

# 30 sentences across 3 topics
documents = [
    # Technology (0)
    "Deep learning models are transforming computer vision tasks.",
    "Neural networks require large amounts of training data.",
    "GPU acceleration speeds up model training significantly.",
    "Transformer architectures dominate natural language processing.",
    "Convolutional layers extract spatial features from images.",
    "Gradient descent optimises neural network weights iteratively.",
    "Attention mechanisms allow models to focus on relevant tokens.",
    "Batch normalisation stabilises and accelerates model training.",
    "Recurrent networks model sequential and time-series data.",
    "Backpropagation computes gradients through the network layers.",
    # Sports (1)
    "The football team won the championship match last Sunday.",
    "Athletes train hard every day to improve their performance.",
    "The basketball player scored thirty points in the final game.",
    "Track and field events include sprinting and jumping disciplines.",
    "Swimming competitions require excellent cardiovascular endurance.",
    "Tennis matches can last several hours in grand slam tournaments.",
    "The coach develops strategies to maximise the team's strengths.",
    "Cycling races cover hundreds of kilometres across mountain terrain.",
    "Marathon runners need months of preparation and endurance training.",
    "The referee enforces the rules and ensures fair play throughout.",
    # Cooking (2)
    "Sautéing vegetables in olive oil enhances their natural flavours.",
    "A pinch of salt balances the sweetness in baked goods.",
    "Slow cooking tenderises tough cuts of meat over several hours.",
    "Fresh herbs like basil and thyme add aroma to pasta dishes.",
    "Caramelising onions requires patience and low consistent heat.",
    "Baking bread at home produces a satisfying crust and crumb.",
    "Emulsifying oil and vinegar creates a smooth salad dressing.",
    "Blanching vegetables preserves their colour and stops cooking.",
    "Knife skills are fundamental to efficient meal preparation.",
    "Fermentation transforms milk into yoghurt cheese and butter.",
]

topic_labels = [0] * 10 + [1] * 10 + [2] * 10
topic_names  = {0: 'Technology', 1: 'Sports', 2: 'Cooking'}

# ── TF-IDF vectorisation (bag-of-words representation)
vectorizer = TfidfVectorizer(stop_words='english', max_features=100)
X_tfidf = vectorizer.fit_transform(documents).toarray()
print(f"TF-IDF matrix shape: {X_tfidf.shape} ({len(documents)} docs, {X_tfidf.shape[1]} terms)")

# ── Apply UMAP with cosine metric (appropriate for TF-IDF unit vectors)
if UMAP_AVAILABLE:
    print("Running UMAP with cosine metric on TF-IDF features...")
    umap_text = umap.UMAP(n_components=2,
                          n_neighbors=5,        # small n because corpus is tiny (30 docs)
                          min_dist=0.1,
                          metric='cosine',       # cosine distance is standard for text
                          random_state=42)
    X_text_umap = umap_text.fit_transform(X_tfidf)
    embed_name = 'UMAP (cosine metric)'
else:
    print("UMAP not available — using t-SNE as proxy (note: t-SNE uses Euclidean by default)")
    X_text_umap = TSNE(n_components=2, perplexity=8, random_state=42, n_iter=500).fit_transform(X_tfidf)
    embed_name = 't-SNE proxy (install umap-learn for cosine metric support)'

# ── Plot
topic_colors = ['steelblue', 'coral', 'seagreen']

fig, ax = plt.subplots(figsize=(9, 7))

for topic_id, (color, name) in enumerate(zip(topic_colors, topic_names.values())):
    mask = np.array(topic_labels) == topic_id
    ax.scatter(X_text_umap[mask, 0], X_text_umap[mask, 1],
               c=color, label=name, s=80, alpha=0.85, edgecolors='white', linewidths=0.5)

# Annotate every point with the first 4 words of the document
for i, doc in enumerate(documents):
    short = ' '.join(doc.split()[:3]) + '...'
    ax.annotate(short, (X_text_umap[i, 0], X_text_umap[i, 1]),
                fontsize=6.5, alpha=0.75,
                xytext=(3, 3), textcoords='offset points')

ax.set_title(f'Text Topic Clustering with {embed_name}\n'
             '(30 sentences across 3 topics)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=11, markerscale=1.5)
ax.set_xticks([])
ax.set_yticks([])
plt.tight_layout()
plt.show()

print()
print("The three topic clusters (Technology, Sports, Cooking) should be well-separated.")
if UMAP_AVAILABLE:
    print("Cosine metric is ideal for TF-IDF vectors because it measures directional similarity,")
    print("ignoring document length differences — a key property of text representations.")

## Self-Check Questions

Test your understanding before moving on.

---

**Question 1:** You have a dataset of 500,000 customer purchase histories (each customer represented by a 200-dimensional embedding). You want to visualise the customer space and also embed new customers into the same map as they sign up. Should you use t-SNE or UMAP? Why?

<details><summary>Show answer</summary>

**UMAP** is the right choice for two reasons:

1. **Scale:** 500,000 points is too large for Barnes-Hut t-SNE (which is O(n² log n) and would take hours). UMAP runs in approximately O(n^1.14) and can handle this in minutes.

2. **Out-of-sample embedding:** t-SNE has no `transform()` method — you cannot embed a new customer without rerunning t-SNE on the entire dataset. UMAP's `transform()` method can embed new customers approximately in seconds by finding their nearest neighbours in the training set.

The practical pipeline: fit `umap.UMAP(n_neighbors=15, min_dist=0.1)` on the 500k historical customers, save the fitted model, and call `reducer.transform(X_new_customer)` for every new sign-up.

</details>

---

**Question 2:** You run UMAP with `n_neighbors=5` and see your data fragment into many small islands. You increase `n_neighbors` to 200 and everything merges into one blob. What value should you try next, and what does this situation tell you about your data?

<details><summary>Show answer</summary>

Try intermediate values: start with `n_neighbors=15` (default), then try 30 and 50. The fact that n_neighbors=5 fragments and n_neighbors=200 merges suggests your data has **hierarchical structure** — there are genuine local sub-groups within larger macro-groups.

At n_neighbors=15-30, you should see a middle ground where the main cluster structure is visible. Importantly, try multiple values and look for structures that are **consistent** — those are the most reliable.

This situation (fragmentation at low n_neighbors, merging at high n_neighbors) is actually informative: it tells you the data has structure at multiple scales, which is useful for deciding how many clusters to target in KMeans.

</details>

---

**Question 3:** A colleague says: "I ran UMAP on our customer data and cluster A is twice as far from cluster B as it is from cluster C in the UMAP plot, so A and B are twice as different." Is this a valid interpretation? What would you tell them?

<details><summary>Show answer</summary>

**Partially valid, but overstated.** Here's the nuanced answer:

- **t-SNE:** This interpretation is completely invalid — inter-cluster distances in t-SNE are meaningless.
- **UMAP:** This interpretation is *qualitatively* more reasonable than t-SNE (UMAP does preserve more global structure), but it's still **not quantitatively precise**. You cannot say "twice as far" means "twice as different."

What you *can* say: in UMAP space, the relative ordering of cluster distances tends to correlate with true high-dimensional distances better than t-SNE does. So "A seems somewhat more different from B than from C" might be a reasonable qualitative observation — but "twice as different" is not justified.

For a precise answer, compute actual distances in the original feature space (or in a PCA-reduced space) between cluster centroids.

</details>

## Key Takeaways

1. **UMAP's two-step algorithm:** Build a fuzzy neighbourhood graph in high-D, then optimise a 2D layout that preserves that graph using cross-entropy minimisation.

2. **Three key hyperparameters:**
   - `n_neighbors` (default 15): neighbourhood size — lower = more local, higher = more global
   - `min_dist` (default 0.1): cluster tightness — lower = compact clusters, higher = spread
   - `metric`: distance function — cosine for text/NLP, euclidean for numeric, hamming for binary

3. **UMAP vs. t-SNE:**
   - Speed: UMAP is 5-10× faster, scales to millions of points
   - Global structure: UMAP preserves it better (cluster distances are *somewhat* meaningful)
   - Out-of-sample: UMAP has `transform()`, t-SNE does not
   - Quality: Both produce excellent local cluster structure for visualisation

4. **UMAP as a feature extractor:** Unlike t-SNE, UMAP can be used as a preprocessing step because new points can be transformed. The UMAP→KMeans pipeline is a valid (if imperfect) approach for retail segmentation.

5. **Metric flexibility:** UMAP's support for cosine and other distance metrics makes it the natural choice for NLP data, where Euclidean distance is inappropriate for high-dimensional sparse vectors.

---

## Week 11 Summary: The Dimensionality Reduction Toolkit

You now have three powerful tools and the wisdom to choose between them:

| | PCA | t-SNE | UMAP |
|-|-----|-------|------|
| **Type** | Linear | Non-linear | Non-linear |
| **Global structure** | ✅ Best | ❌ Distorted | ✅ Good |
| **Local clusters** | ⚠️ OK | ✅ Excellent | ✅ Excellent |
| **Speed** | ✅ Fastest | ⚠️ Slow | ✅ Fast |
| **New data** | ✅ Yes | ❌ No | ✅ Yes |
| **Interpretable** | ✅ Loadings | ❌ No | ❌ No |
| **Deterministic** | ✅ Yes | ❌ No | ⚠️ Mostly |

**The practical workflow for customer segmentation:**
1. Standardise → PCA(20-50) to remove noise and speed things up
2. Run KMeans (or DBSCAN) in PCA space to find actual clusters
3. Use t-SNE or UMAP *only* to visualise — colour points by cluster label
4. If you need to embed new customers: UMAP with `transform()`